Các bước thực hiện trong colab này:
+ Đọc dữ liệu
+ Clean data
+ Xử lý outlier bằng Zscore
+ Cho vào SMOTE
+ MFCM
+ Chia dữ liệu train, test
+ Scale tập train, test
+ Cho vào mô hình
- tỉ lệ trên tập test cân bằng là

In [206]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [207]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [208]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [209]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [210]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1115
1.0,846
-1.0,165


In [211]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1655
2.0,295
3.0,176


Clean data

In [212]:
data.duplicated().sum()

np.int64(13)

In [213]:
data.drop_duplicates(inplace=True)

In [214]:
data.duplicated().sum()

np.int64(0)

In [215]:
data.isna().sum().sum()

np.int64(0)

In [216]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
baseline value,2113.0,133.304780,9.837451,106.0,126.000,133.000,140.000,160.000
accelerations,2113.0,0.003188,0.003871,0.0,0.000,0.002,0.006,0.019
fetal_movement,2113.0,0.009517,0.046804,0.0,0.000,0.000,0.003,0.481
uterine_contractions,2113.0,0.004387,0.002941,0.0,0.002,0.005,0.007,0.015
light_decelerations,2113.0,0.001901,0.002966,0.0,0.000,0.000,0.003,0.015
severe_decelerations,2113.0,0.000003,0.000057,0.0,0.000,0.000,0.000,0.001
prolongued_decelerations,2113.0,0.000159,0.000592,0.0,0.000,0.000,0.000,0.005
abnormal_short_term_variability,2113.0,46.993848,17.177782,12.0,32.000,49.000,61.000,87.000
mean_value_of_short_term_variability,2113.0,1.335021,0.884368,0.2,0.700,1.200,1.700,7.000
percentage_of_time_with_abnormal_long_term_variability,2113.0,9.795078,18.337073,0.0,0.000,0.000,11.000,91.000


Outlier Zscore

In [217]:
from scipy.stats import zscore
import numpy as np
import pandas as pd

# Tách cột đầu vào và cột đầu ra
X = data.drop(columns=['fetal_health'])
y = data['fetal_health']

# Chỉ lấy các cột số trong X
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Tính Z-score cho các cột số
z_scores = np.abs(zscore(X[num_cols], nan_policy='omit'))

# Nếu chỉ có 1 cột số thì đưa về DataFrame cho đồng nhất
if len(num_cols) == 1:
    z_scores = pd.DataFrame(z_scores, columns=num_cols, index=X.index)
else:
    z_scores = pd.DataFrame(z_scores, columns=num_cols, index=X.index)

# Đặt ngưỡng Z-score
threshold = 3.5

# Giữ lại các dòng không có outlier
mask = (z_scores < threshold).all(axis=1)

X_clean = X[mask]
y_clean = y[mask]

# Ghép lại thành dataframe mới nếu cần
data_clean = pd.concat([X_clean, y_clean], axis=1)

print("Kích thước trước khi xử lý:", data.shape)
print("Kích thước sau khi xử lý:", data_clean.shape)

Kích thước trước khi xử lý: (2113, 22)
Kích thước sau khi xử lý: (1920, 22)


MFCM

Tính trọng số

In [218]:
import numpy as np
import math

# 1. Hàm tính entropy cho Binary
def compute_binary_entropy(data, col_idx, n):
    p_b = np.mean(data[:, col_idx])  # Xác suất giá trị 1
    if p_b == 0 or p_b == 1:
        return 0
    return 0.5 * np.log(2 * np.pi * np.e * n * p_b * (1 - p_b))

# 2. Hàm tính entropy cho Nominal
def compute_nominal_entropy(data, col_idx, n):
    counts = np.unique(data[:,col_idx], return_counts=True) # trả về mảng các giá trị được đếm
    p_ri = counts[1] / data[:,col_idx].shape[0] # Sửa lại shape[0] thay vì shape để lấy số lượng mẫu
    p_ri_log = np.log(p_ri)

    return (((len(counts[1]) - 1) / 2) * np.log(2 * np.pi * np.e * n)) + 0.5 * np.sum(p_ri_log)

# 3. Hàm tính entropy cho Ordinal
def compute_ordinal_entropy(data, col_idx, n):
    H_nom = compute_nominal_entropy(data, col_idx, n)
    L_max = np.max(data[:,col_idx])
    L_min = np.min(data[:, col_idx])

    R_o = (L_max - L_min)
    R_o = math.ceil(R_o)

    term = 0
    for j in range(1, int(R_o)+1):
        term += np.log(math.comb(int(R_o) - 1, j - 1))

    term = term / R_o if R_o != 0 else 0
    return H_nom - np.log(R_o) + term - (R_o - 1) / 2.0

# 4. Hàm tính entropy cho Interval
def compute_interval_entropy(data, col_idx, n):
    n_bins = round(1 + 1.3*np.log(n)) # Dùng công thức Sturges hoặc tương tự để chia bin

    # Tính histogram
    hist, bin_edges = np.histogram(data[:, col_idx], bins=n_bins)
    p_ri = hist / len(data[:, col_idx])
    p_ri = p_ri[p_ri > 0]  # Loại bỏ xác suất 0 để không bị lỗi log(0)

    return ((n_bins - 1) / 2) * np.log(2 * np.pi * np.e * n) + 0.5 * np.sum(np.log(p_ri))

Khoảng cách MFCM

In [219]:
def compute_distances(DataCV, V, Binary_Col, Nominal_Col, Ordinal_Col, Interval_Col, w_B, w_N, w_O, w_S, p=2):
    n = DataCV.shape[0]
    c = V.shape[0]
    distances = np.zeros((n, 4, c))

    # 1. Khoảng cách Binary
    for col_index in Binary_Col:
        for i in range(n):
            for k in range(c):
                v_binary = 1 if V[k, col_index] >= 0.5 else 0
                distances[i, 0, k] += (DataCV[i, col_index] != v_binary)

    # 2. Khoảng cách Nominal
    for col_index in Nominal_Col:
        for i in range(n):
            for k in range(c):
                v_nominal = np.round(V[k, col_index])
                distemp = 0
                if (DataCV[i, col_index] == v_nominal):
                    distemp = 1
                distances[i, 1, k] += (1 - distemp)

    # 3. Tính khoảng cách Ordinal
    for i in range(n):
        for k in range(c):
            ordinal_dist = 0
            for col_index in Ordinal_Col:
                values = DataCV[:, col_index].astype(int)
                L_max = np.max(values)
                L_min = np.min(values)
                R_o = L_max - L_min
                R_o = 1 if R_o == 0 else R_o # Tránh lỗi chia cho 0
                ordinal_dist += (np.abs(DataCV[i, col_index] - V[k, col_index]) / R_o) ** p
            distances[i, 2, k] += (ordinal_dist ** (1/p))

    # 4. Tính khoảng cách Interval
    for col_index in Interval_Col:
        for i in range(n):
            for k in range(c):
                distances[i, 3, k] += (DataCV[i, col_index] - V[k, col_index]) ** 2

    # 5. Tổng hợp khoảng cách nhân với trọng số
    dV = np.zeros((n, c))
    for i in range(n):
        for k in range(c):
            dV[i, k] = (distances[i, 0, k] * w_B +
                        distances[i, 1, k] * w_N +
                        distances[i, 2, k] * w_O +
                        distances[i, 3, k] * w_S)
    return dV

Tính trọng số MFCM

In [220]:
def compute_weights(data, binary_cols, interval_cols):
    n = data.shape[0]

    # ===== Binary =====
    H_B_list = []
    for col in binary_cols:
        H_B_list.append(compute_binary_entropy(data, col, n))
    H_B = np.mean(H_B_list) if len(H_B_list) > 0 else 0

    # ===== Interval =====
    H_S_list = []
    for col in interval_cols:
        H_S_list.append(compute_interval_entropy(data, col, n))
    H_S = np.mean(H_S_list) if len(H_S_list) > 0 else 0

    # ===== Tổng =====
    H_T = H_B + H_S

    # Tránh chia 0
    if H_T == 0:
        return 0.5, 0.5

    # ===== Weight =====
    w_B = H_B / H_T
    w_S = H_S / H_T

    return w_B, w_S

Train MFCM

In [221]:
def train_mfcm(DataCV, U_init, V_init, Binary_Col, Nominal_Col, Ordinal_Col, Interval_Col,
               W_b, W_n, W_o, W_i, m=2, c=2, max_iter=50, delta=0.01, p=2):

    n = DataCV.shape[0]
    U = U_init.copy()
    V = V_init.copy()
    # cập nhật tâm cụm ngay lúc bắt đầu theo U(ma trận membership)
    for k in range(c):
          # Tính tử số và mẫu số cho công thức cập nhật tâm V
          V[k] = np.sum(DataCV * (U[:, k, np.newaxis] ** m), axis=0) / np.sum(U[:, k] ** m)

    for iteration in range(max_iter):
        # Bước 1: Tính ma trận khoảng cách
        distances = compute_distances(DataCV, V, Binary_Col, Nominal_Col, Ordinal_Col, Interval_Col,
                                      W_b, W_n, W_o, W_i, p)
        # Bước 2: Cập nhật mức độ thành viên U
        U_new = np.zeros((n, c))

        for i in range(n):

            # Nếu có cụm nào distance = 0
            if np.any(distances[i] == 0):
                zero_index = np.where(distances[i] == 0)[0]
                U_new[i, zero_index] = 1
                continue

            for k in range(c):
                denom = np.sum((distances[i, k] / distances[i, :]) ** (2 / (m - 1)))
                U_new[i, k] = 1 / denom

        # Bước 3: Kiểm tra hội tụ
        if np.max(np.abs(U_new - U)) < delta:
            print(f"Thuật toán hội tụ sau {iteration + 1} vòng lặp")
            U = U_new
            break

        U = U_new

        # Bước 4: Cập nhật trung tâm cụm V
        for k in range(c):
            # Tính tử số và mẫu số cho công thức cập nhật tâm V
            V[k] = np.sum(DataCV * (U[:, k, np.newaxis] ** m), axis=0) / np.sum(U[:, k] ** m)

    return U, V

In [222]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score
)

target_col = 'fetal_health'

print("Kích thước data gốc:", data_clean.shape)
display(data_clean.head())

# =============================
# 1. Tách X, y từ toàn bộ data
# =============================
X = data_clean.drop(columns=[target_col]).copy()
y = data_clean[target_col].copy().astype(int)

print("\nPhân bố lớp ban đầu:")
print(y.value_counts().sort_index())

# =============================
# 2. SMOTE trên toàn bộ data
# =============================
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42,
    k_neighbors=5
)

X_smote, y_smote = smote.fit_resample(X, y)

X_smote = pd.DataFrame(X_smote, columns=X.columns)
y_smote = pd.Series(y_smote, name=target_col)

print("\nPhân bố lớp sau SMOTE toàn bộ data:")
print(y_smote.value_counts().sort_index())

print("\nKích thước sau SMOTE:")
print("X_smote:", X_smote.shape)
print("y_smote:", y_smote.shape)

# =============================
# 3. Chia train/test sau khi SMOTE
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X_smote,
    y_smote,
    test_size=0.2,
    random_state=42,
    stratify=y_smote
)

print("\nPhân bố y_train:")
print(y_train.value_counts().sort_index())

print("\nPhân bố y_test:")
print(y_test.value_counts().sort_index())

# =============================
# 4. Scale sau khi chia
# =============================
scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("\nĐã scale xong dữ liệu")

# =========================================================
# 5. MFCM: train trên train set và lấy membership làm feature
# =========================================================

# Chuyển sang numpy
DataCV_train = X_train_scaled.values
DataCV_test = X_test_scaled.values

# Số cụm
#4 đang thấy ổn
c = 4
m = 2
n_train = DataCV_train.shape[0]

# Khởi tạo U ngẫu nhiên
np.random.seed(42)
U_init = np.random.dirichlet(np.ones(c), size=n_train)

# Khởi tạo tâm cụm ngẫu nhiên từ train
idx = np.random.choice(n_train, c, replace=False)
V_init = DataCV_train[idx].copy()

# Với bộ fetal_health đã scale, đơn giản nhất coi tất cả là interval
Binary_Col = []
Nominal_Col = []
Ordinal_Col = []
Interval_Col = list(range(DataCV_train.shape[1]))

# Tính trọng số
W_b, W_i = compute_weights(DataCV_train, Binary_Col, Interval_Col)
W_n = 0
W_o = 0

print("\nTrọng số MFCM:")
print("W_b =", W_b)
print("W_i =", W_i)
print("W_n =", W_n)
print("W_o =", W_o)

# Train MFCM trên tập train
U_train_mfcm, V_final = train_mfcm(
    DataCV=DataCV_train,
    U_init=U_init,
    V_init=V_init,
    Binary_Col=Binary_Col,
    Nominal_Col=Nominal_Col,
    Ordinal_Col=Ordinal_Col,
    Interval_Col=Interval_Col,
    W_b=W_b,
    W_n=W_n,
    W_o=W_o,
    W_i=W_i,
    m=m,
    c=c,
    max_iter=50,
    delta=0.01,
    p=2
)

# Hàm suy ra membership cho test từ tâm cụm đã học
def predict_mfcm_membership(DataCV, V, Binary_Col, Nominal_Col, Ordinal_Col, Interval_Col,
                            W_b, W_n, W_o, W_i, m=2, p=2):
    distances = compute_distances(
        DataCV, V,
        Binary_Col, Nominal_Col, Ordinal_Col, Interval_Col,
        W_b, W_n, W_o, W_i, p
    )

    n = DataCV.shape[0]
    c = V.shape[0]
    U_pred = np.zeros((n, c))

    for i in range(n):
        if np.any(distances[i] == 0):
            zero_index = np.where(distances[i] == 0)[0]
            U_pred[i, zero_index] = 1
            continue

        for k in range(c):
            denom = np.sum((distances[i, k] / distances[i, :]) ** (2 / (m - 1)))
            U_pred[i, k] = 1 / denom

    return U_pred

# Membership cho test
U_test_mfcm = predict_mfcm_membership(
    DataCV=DataCV_test,
    V=V_final,
    Binary_Col=Binary_Col,
    Nominal_Col=Nominal_Col,
    Ordinal_Col=Ordinal_Col,
    Interval_Col=Interval_Col,
    W_b=W_b,
    W_n=W_n,
    W_o=W_o,
    W_i=W_i,
    m=m,
    p=2
)

# Tạo DataFrame membership
mfcm_cols = [f"mfcm_cluster_{i+1}" for i in range(c)]

U_train_df = pd.DataFrame(U_train_mfcm, columns=mfcm_cols, index=X_train_scaled.index)
U_test_df = pd.DataFrame(U_test_mfcm, columns=mfcm_cols, index=X_test_scaled.index)

print("\nMembership train:")
display(U_train_df.head())

print("\nMembership test:")
display(U_test_df.head())

# Ghép membership vào feature gốc
X_train_final = pd.concat([X_train_scaled, U_train_df], axis=1)
X_test_final = pd.concat([X_test_scaled, U_test_df], axis=1)

print("\nKích thước sau khi thêm membership:")
print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)

display(X_train_final.head())

# =============================
# 6. Train model
# =============================
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train_final, y_train)
y_pred = rf.predict(X_test_final)

# =============================
# 7. Đánh giá
# =============================
print("\n=== KẾT QUẢ ===")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nKQ FOR EXCEL\n")
print("Precision weighted:", precision_score(y_test, y_pred, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred, average='macro'))

print("\nDetail about one class:")
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nConfusion Matrix (normalize='true'):")
print(confusion_matrix(y_test, y_pred, normalize='true'))

Kích thước data gốc: (1920, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0



Phân bố lớp ban đầu:
fetal_health
1    1554
2     279
3      87
Name: count, dtype: int64

Phân bố lớp sau SMOTE toàn bộ data:
fetal_health
1    1554
2    1554
3    1554
Name: count, dtype: int64

Kích thước sau SMOTE:
X_smote: (4662, 21)
y_smote: (4662,)

Phân bố y_train:
fetal_health
1    1243
2    1243
3    1243
Name: count, dtype: int64

Phân bố y_test:
fetal_health
1    311
2    311
3    311
Name: count, dtype: int64

Đã scale xong dữ liệu

Trọng số MFCM:
W_b = 0.0
W_i = 1.0
W_n = 0
W_o = 0
Thuật toán hội tụ sau 30 vòng lặp

Membership train:


,mfcm_cluster_1,mfcm_cluster_2,mfcm_cluster_3,mfcm_cluster_4
976,0.048641,0.090849,0.387117,0.473393
675,0.004874,0.424155,0.533178,0.037793
4635,0.926217,0.012964,0.020388,0.040431
501,0.144560,0.041407,0.107935,0.706098
3614,0.001545,0.966574,0.024624,0.007257



Membership test:


,mfcm_cluster_1,mfcm_cluster_2,mfcm_cluster_3,mfcm_cluster_4
3476,0.906869,0.017612,0.024292,0.051227
109,0.014553,0.295152,0.585348,0.104948
2100,0.010067,0.703938,0.200391,0.085604
4365,0.007072,0.107115,0.854964,0.030849
3943,0.952608,0.008130,0.013041,0.026221



Kích thước sau khi thêm membership:
X_train_final: (3729, 25)
X_test_final : (933, 25)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,mfcm_cluster_1,mfcm_cluster_2,mfcm_cluster_3,mfcm_cluster_4
976,0.296296,0.2500,0.000000,0.214286,0.000000,0.0,0.000000,0.120394,0.357143,0.000000,...,0.500000,0.400000,0.460784,0.376344,0.050420,0.500000,0.048641,0.090849,0.387117,0.473393
675,0.500000,0.0000,0.100719,0.000000,0.000000,0.0,0.000000,0.802626,0.023810,0.273973,...,0.000000,0.470000,0.549020,0.462366,0.025210,0.500000,0.004874,0.424155,0.533178,0.037793
4635,0.188346,0.0718,0.005510,0.367771,0.686178,0.0,0.691467,0.707845,0.519746,0.000000,...,0.617067,0.148731,0.093663,0.090374,0.331482,0.000000,0.926217,0.012964,0.020388,0.040431
501,0.407407,0.0625,0.000000,0.500000,0.500000,0.0,0.000000,0.361182,0.595238,0.000000,...,0.000000,0.470000,0.441176,0.397849,0.201681,0.500000,0.144560,0.041407,0.107935,0.706098
3614,0.638385,0.0000,0.019686,0.000000,0.000000,0.0,0.000000,0.858933,0.023810,0.986301,...,0.000000,0.573849,0.611617,0.552526,0.000000,0.956067,0.001545,0.966574,0.024624,0.007257



=== KẾT QUẢ ===
Accuracy: 0.9860664523043944

KQ FOR EXCEL

Precision weighted: 0.9860793251553432
Recall weighted: 0.9860664523043944
F1 weighted: 0.9860506655169249
Precision macro: 0.9860793251553434
Recall macro: 0.9860664523043944
F1 macro: 0.9860506655169249

Detail about one class:
Balanced Accuracy: 0.9860664523043944
Macro F1: 0.9860506655169249

Classification Report:
              precision    recall  f1-score   support

           1     0.9902    0.9775    0.9838       311
           2     0.9776    0.9807    0.9791       311
           3     0.9904    1.0000    0.9952       311

    accuracy                         0.9861       933
   macro avg     0.9861    0.9861    0.9861       933
weighted avg     0.9861    0.9861    0.9861       933


Confusion Matrix:
[[304   7   0]
 [  3 305   3]
 [  0   0 311]]

Confusion Matrix (normalize='true'):
[[0.97749196 0.02250804 0.        ]
 [0.0096463  0.9807074  0.0096463 ]
 [0.         0.         1.        ]]


XGBoot

In [223]:
y_train = y_train.astype(int) - 1
y_test = y_test.astype(int) - 1

In [224]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Khởi tạo model XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',   # dùng cho phân loại nhiều lớp
    num_class=len(y_train.unique()),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

# Train model
xgb_model.fit(X_train_final, y_train)

# Predict
y_pred_xg = xgb_model.predict(X_test_final)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred_xg))
print("Precision weighted:", precision_score(y_test, y_pred_xg, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred_xg, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred_xg, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred_xg, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred_xg, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred_xg, average='macro'))

print("\nDetail about one class:")
print("Classification Report:\n", classification_report(y_test, y_pred_xg))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xg))

Accuracy: 0.9871382636655949
Precision weighted: 0.9872631570546078
Recall weighted: 0.9871382636655949
F1 weighted: 0.9871478150602928
Precision macro: 0.9872631570546075
Recall macro: 0.9871382636655949
F1 macro: 0.9871478150602928

Detail about one class:
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99       311
           1       0.97      0.99      0.98       311
           2       0.99      0.99      0.99       311

    accuracy                           0.99       933
   macro avg       0.99      0.99      0.99       933
weighted avg       0.99      0.99      0.99       933


Confusion Matrix:
 [[304   6   1]
 [  1 308   2]
 [  0   2 309]]
